# nextvit fold0/fold1 재학습 (Colab)
distributed 실험에서 nextvit가 fold0·fold1만 학습 실패(acc 0.500) → lr 1e-4로 재학습.
Drive에 기존 가중치가 있어야 함(colab_fold3 실행본). **T4/L4 GPU** 설정 후 모두 실행.
nextvit만 2개 fold → ~30~40분.

In [ ]:
# Cell 1: GPU + 클론
import torch
assert torch.cuda.is_available(), 'GPU 런타임으로 변경'
print('GPU:', torch.cuda.get_device_name(0))
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# Cell 2: Drive 연결 (기존 가중치 + 결과)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
CKPT = '/content/drive/MyDrive/fireimage_dist_ckpt'
for sub in ['model_save', 'results']:
    os.makedirs(f'{CKPT}/{sub}', exist_ok=True)
    link = f'/content/fireimage_detection/{sub}'
    if os.path.islink(link): os.unlink(link)
    elif os.path.exists(link): shutil.rmtree(link, ignore_errors=True)
    os.symlink(f'{CKPT}/{sub}', link)
import glob
print('기존 nextvit 가중치:', glob.glob(f'{CKPT}/model_save/**/nextvit*.pt', recursive=True))

In [ ]:
# Cell 3: 패키지
!pip install timm einops transformers yt-dlp kaggle -q

In [ ]:
# Cell 4: Kaggle 인증
import os, getpass, re
raw = getpass.getpass('Kaggle API Token (KGAT_...): ')
token = re.sub(r'[^A-Za-z0-9_\-]', '', raw)
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'),'w').write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
!kaggle datasets list --user yuntarwon

In [ ]:
# Cell 5: 데이터 (video1 + 비youtube + video2 재추출) — distributed와 동일해야 split 일치
import os, glob, zipfile
BASE='/content/fireimage_detection/data/fireimage'
def dl(ds,dst):
    os.makedirs(dst,exist_ok=True); os.system(f'kaggle datasets download {ds} -p {dst} --unzip')
    for _ in range(2):
        zs=glob.glob(f'{dst}/**/*.zip',recursive=True)
        if not zs: break
        for z in zs:
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(dst)
                os.remove(z)
            except: pass
dl('yuntarwon/fireimage-abnormal',f'{BASE}/abnormal')
dl('yuntarwon/fireimage-abnormal-youtube',f'{BASE}/abnormal/youtube')
dl('yuntarwon/fireimage-normal',f'{BASE}/normal')
!python data/youtube_preprocessor.py --url "https://www.youtube.com/watch?v=eEP8a2u5PbA" --subdir youtube2 --sample_every 8 --max_abnormal 600 --max_normal 300
!python data/youtube_preprocessor.py --stats

In [ ]:
# Cell 6: nextvit fold0/fold1 재학습 (lr 1e-4)
%cd /content/fireimage_detection
!python main_nextvit_fix.py --class_name fireimage

In [ ]:
# Cell 7: 결과 확인 (nextvit_0 / nextvit_1 갱신 확인)
import pandas as pd
df = pd.read_csv('/content/fireimage_detection/results/fireimage_dist/metrics.csv')
print(df[df['model name'].str.startswith('nextvit')].to_string())

In [ ]:
# Cell 8: 갱신된 결과 다운로드
import shutil
shutil.make_archive('/content/dist_results', 'zip', '/content/fireimage_detection', 'results/fireimage_dist')
from google.colab import files
files.download('/content/dist_results.zip')